# Data Profiling

## 1. Datensatz (Überblick)

Der Datensatz liegt als Excel-Arbeitsmappe mit zwei Arbeitsblättern vor. Bevor das eigentliche Profiling beginnt, wird geprüft, ob beide Arbeitsblätter gemeinsam analysiert werden können.

In [ ]:
from src.paths import RAW_DATA_DIR
import pandas as pd
import re

xlsx_file = RAW_DATA_DIR / "online_retail_II.xlsx"

sheet_1 = pd.read_excel(xlsx_file, sheet_name="Year 2009-2010")
sheet_2 = pd.read_excel(xlsx_file, sheet_name="Year 2010-2011")

## 2. Struktur der Arbeitsblätter

Die Struktur beider Arbeitsblätter wird verglichen. Dazu werden die Spaltennamen, deren Reihenfolge sowie die Datentypen betrachtet.

In [ ]:
print([sheet.info() for sheet in [sheet_1, sheet_2]])

Beide Arbeitsblätter besitzen dieselben Spalten in identischer Reihenfolge sowie kompatible Datentypen.
Daher werden sie für das weitere Profiling zu einem gemeinsamen DataFrame zusammengeführt.

In [ ]:
dataset = pd.concat([sheet_1, sheet_2], ignore_index=True)

## 3. Spaltenanalyse

Die Spalten werden hinsichtlich ihrer Datentypen, fehlender Werte und möglicher Identifikatoren untersucht.

Darüber hinaus werden auffällige Ausprägungen der Spalten **StockCode** und **Description** näher betrachtet. Ein Teil dieser Untersuchung erfolgt anhand exportierter Übersichten,
die in Excel manuell geprüft werden.

In [ ]:
print(dataset.info())

Die Spalten **Invoice** und **StockCode** enthalten Identifikatoren, **Description** Textinformationen.
Diese Spalten werden für die weitere Analyse in den String-Datentyp konvertiert.
Dadurch bleibt der Informationsgehalt unverändert, während Vergleiche und Sortierungen konsistent durchgeführt werden können.

In [ ]:
dataset["Invoice"] = dataset["Invoice"].astype("string")
dataset["StockCode"] = dataset["StockCode"].astype("string")
dataset["Description"] = dataset["Description"].astype("string")

print(dataset.info())

Die Datentypen entsprechen nun dem Inhalt der Spalten und bilden die Grundlage für die folgenden Analysen.

In [ ]:
print(dataset.isna().sum(), end="\n\n")
print(f"Invoice (is unique): {dataset["Invoice"].is_unique}\n")
print(f"[Invoice, StockCode] (is unique): {len(dataset[["Invoice", "StockCode"]].drop_duplicates()) == len(dataset)}")

Die Spalten **Description** und **Customer ID** enthalten fehlende Werte.
Weder **Invoice** noch die Kombination aus **Invoice und StockCode** identifizieren eine Zeile eindeutig.
Dasselbe Produkt kann innerhalb einer Rechnung mehrfach als separate Position vorkommen, obwohl die Menge in **Quantity** erfasst wird.
Ein eindeutiger Schlüssel auf Positionsebene ist nicht vorhanden.

### Zuordnung von StockCode und Description

**StockCode** dient als Produktidentifikation, während **Description** die zugehörige Produktbezeichnung enthält. Daher wird geprüft, ob jedem **StockCode**
genau eine **Description** zugeordnet ist.

In [ ]:
description_count = dataset.groupby("StockCode")["Description"].nunique()

print(f"StockCodes with multiple Descriptions: {(description_count > 1).sum():,}")

Einzelne **StockCodes** kommen mit mehreren unterschiedlichen **Descriptions** vor. Eine eindeutige Zuordnung zwischen **StockCode** und **Description** ist
daher nicht gegeben.

Da anhand der vorhandenen Daten nicht bestimmt werden kann, welche der unterschiedlichen Bezeichnungen fachlich die korrekte ist, wird **StockCode** weiterhin
als maßgebliche Produktidentifikation betrachtet.

In [ ]:
print(dataset["Invoice"].value_counts())
print(dataset["Invoice"].value_counts().mean())
print(dataset["Invoice"].value_counts().median())

counts = dataset["Invoice"].value_counts()
for idx in counts[counts > 1000].index:
    idx_loc = dataset["Invoice"] == idx
    print(dataset.loc[idx_loc, "Customer ID"].count())
    print(dataset.loc[idx_loc, "Country"].unique())
    print((dataset.loc[idx_loc, "Quantity"] < 0).sum())
    print((dataset.loc[idx_loc, "Price"] < 0).sum())

display(dataset[dataset["Invoice"] == 537434].sort_values(by="StockCode").head(30))

display(
    dataset[dataset.duplicated(keep=False)]
    .sort_values(by=["Invoice", "StockCode"])
)

Die Anzahl der Positionen pro Rechnung variiert deutlich. Während die meisten Rechnungen nur wenige Positionen enthalten, existieren einzelne Rechnungen
mit mehr als 1.000 Positionen.

Negative Mengen oder Preise könnten auf Storno-, Rückgabe- oder Korrekturbuchungen hindeuten. Die untersuchten Rechnungen mit mehr als 1.000 Positionen
weisen jedoch weder negative Mengen noch negative Preise auf. Darüber hinaus enthalten sie keine **Customer ID**, stammen ausschließlich aus dem Vereinigten
Königreich und beginnen nicht mit dem Präfix `C`, das im Datensatz für Stornorechnungen verwendet wird. Die fachliche Bedeutung dieser Rechnungen lässt
sich anhand der verfügbaren Informationen nicht eindeutig bestimmen.

Bei der Detailanalyse dieser Rechnungen fallen vollständig identische Rechnungspositionen auf. Eine anschließende Untersuchung des gesamten Datensatzes
zeigt, dass 67.242 Zeilen Teil vollständig identischer Datensätze sind. Davon stellen 34.335 Zeilen redundante Wiederholungen dar.

Da diese Zeilen in sämtlichen Spalten identisch sind, enthalten die zusätzlichen Wiederholungen keine weiteren Informationen.

### Unterschiedliche Zeitstempel innerhalb einer Rechnung

Da eine Rechnung aus mehreren Positionen bestehen kann, wird zusätzlich geprüft, ob **InvoiceDate** innerhalb derselben **Invoice** konsistent ist.
Dazu wird für jede Rechnung die Differenz zwischen dem frühesten und spätesten **InvoiceDate** bestimmt.

In [ ]:
grp = dataset.groupby("Invoice")["InvoiceDate"]

d_dif = grp.max() - grp.min()
d_dif = d_dif[d_dif > pd.Timedelta(0)]

print(f"Maximum difference: {d_dif.max().total_seconds() / 60} minutes")
print(f"Average difference: {d_dif.mean().total_seconds() / 60} minutes")

Bei einzelnen Rechnungen unterscheiden sich die Zeitstempel der Positionen geringfügig. Die durchschnittliche Abweichung liegt bei rund einer Minute, die
maximale Abweichung beträgt neun Minuten.

## 4. Untersuchung vollständiger Duplikate

Nachdem festgestellt wurde, dass keine einzelne Spalte und auch keine Kombination aus **Invoice** und **StockCode** einen Datensatz eindeutig identifiziert, wird
untersucht, ob vollständig identische Datensätze im Datensatz vorkommen.

Dabei wird ermittelt,

- wie viele Zeilen zu vollständigen Duplikatgruppen gehören,
- wie viele zusätzliche Wiederholungen existieren und
- wie viele eindeutige Datensätze der Datensatz insgesamt enthält.

In [ ]:
duplicate_mask = dataset.duplicated(keep=False)

duplicate_rows = duplicate_mask.sum()
redundant_duplicates = dataset.duplicated().sum()
unique_rows = dataset.drop_duplicates().shape[0]

print(f"Rows in duplicate groups : {duplicate_rows:,}")
print(f"Redundant duplicates     : {redundant_duplicates:,}")
print(f"Unique rows              : {unique_rows:,}")

In [ ]:
display(
    dataset[duplicate_mask]
    .sort_values(["Invoice", "StockCode"])
    .head(20)
)

Die Stichprobe zeigt, dass einzelne Datensätze in sämtlichen Spalten übereinstimmen. Anhand der verfügbaren Informationen lässt sich nicht erkennen, wodurch
diese Wiederholungen entstanden sind. Da keine Unterschiede zwischen den Zeilen bestehen, können sie im weiteren Verlauf der Datenaufbereitung gesondert
betrachtet werden.

## 5. Untersuchung von Stornorechnungen

Laut der Datensatzbeschreibung kennzeichnet eine Rechnungsnummer mit dem Präfix `C` eine Stornierung.

Zunächst wird ermittelt, wie viele Positionen zu solchen Rechnungen gehören. Da vollständige Duplikate die Anzahl der betroffenen Positionen erhöhen können,
wird die Anzahl sowohl im ursprünglichen Datensatz als auch nach dem Beibehalten unterschiedlicher Zeilen betrachtet.

In [ ]:
cancellation_mask = dataset["Invoice"].str.startswith("C")
cancellation_rows = cancellation_mask.sum()

distinct_rows = dataset.drop_duplicates()

distinct_cancellation_mask = distinct_rows["Invoice"].str.startswith("C")
distinct_cancellation_rows = distinct_cancellation_mask.sum()

print(f"Cancellation rows              : {cancellation_rows:,}")
print(f"Distinct cancellation rows     : {distinct_cancellation_rows:,}")

In [ ]:
cancellations = distinct_rows[distinct_cancellation_mask]

print(cancellations["Quantity"].lt(0).value_counts())

Nach dem Beibehalten unterschiedlicher Zeilen gehören **19.104 Positionen** zu Rechnungen, deren Rechnungsnummer mit `C` beginnt.

Die Mengen dieser Positionen sind nahezu ausschließlich negativ. Damit stimmen die Daten weitgehend mit der Beschreibung überein, nach der das Präfix `C` eine
Stornierung kennzeichnet.

Es existiert jedoch mindestens eine Position mit einer positiven Menge. Diese besitzt den **StockCode** `M`, die Beschreibung `Manual` und keine **Customer ID**.
Ihre fachliche Bedeutung lässt sich anhand der verfügbaren Informationen nicht eindeutig bestimmen.

## 6. Untersuchung auffälliger StockCodes

### 6.1 Nummerisch beginnende StockCodes

Zunächst werden **StockCodes** betrachtet, die mit einer Ziffer beginnen und mit einem Buchstaben enden.

In [ ]:
postfix = dataset[dataset["StockCode"].str.match(r"^\d+[a-z]", flags=re.IGNORECASE)]
print(postfix.head(10))
print(len(postfix))

# postfix.to_csv(
#     RAW_DATA_DIR / "postfix.csv",
#     index=False
# )

Diese Datensätze wurden zusätzlich exportiert und manuell untersucht.

Die manuelle Durchsicht zeigt, dass der angehängte Buchstabe überwiegend Bestandteil regulärer Produktcodes ist und nicht allgemein zur Kennzeichnung von
Sonderfällen dient.

Innerhalb dieser Gruppe treten zwar auch negative Mengen, fehlende Beschreibungen und weitere Auffälligkeiten auf, diese betreffen jedoch nur einen kleinen
Teil der Datensätze.

Von insgesamt **128.893** Zeilen besitzen lediglich **3.377** eine negative Menge.

In [ ]:
print(f"postfix_pct: {len(postfix) * 100 / len(dataset)}%")

Die Gruppe umfasst rund **12,08 %** des gesamten Datensatzes. Aufgrund der manuellen Untersuchung werden diese **StockCodes** nicht grundsätzlich als
Sonderfälle behandelt.

### 6.2 Nicht numerisch beginnende StockCodes

Anschließend werden **StockCodes** untersucht, die nicht mit einer Ziffer beginnen.

Diese Gruppe ist besonders relevant, da sie neben möglichen Produktcodes auch Bezeichnungen für Versandkosten, Gebühren, Rabatte, manuelle Buchungen und weitere
Sonderfälle enthalten kann.

Für die manuelle Untersuchung werden die **StockCodes** nach ihrer Häufigkeit gruppiert und als CSV-Datei exportiert. Die exportierte Übersicht wird anschließend
in Excel anhand der Codes und ihrer zugehörigen Beschreibungen geprüft.

In [ ]:
business_cases = dataset[dataset["StockCode"].str.match(r"^[^\d]")]

stockcode_counts = (
    business_cases["StockCode"]
    .value_counts()
)

print(f"Rows: {business_cases.shape[0]:,}")
print(f"Unique StockCodes: {business_cases['StockCode'].nunique():,}")

print(stockcode_counts)

# stockcode_counts.to_csv(
#     RAW_DATA_DIR / "non_numeric_stockcode_counts.csv",
#     index=False,
# )

In [ ]:
stockcode_descriptions = (
    business_cases[["StockCode", "Description"]]
    .value_counts(dropna=False)
    .rename("Count")
    .reset_index()
    .sort_values(["StockCode", "Count"], ascending=[True, False])
)

print(stockcode_descriptions)

# stockcode_descriptions.to_csv(
#     RAW_DATA_DIR / "non_numeric_stockcode_descriptions.csv",
#     index=False,
# )

Die exportierten Übersichten wurden in Excel nach **StockCode**, **Description** und Häufigkeit untersucht.

Insgesamt enthält der Datensatz **62 unterschiedliche StockCodes**, die nicht mit einer Ziffer beginnen.
Die manuelle Prüfung zeigt, dass diese Gruppe nicht einheitlich behandelt werden kann: Neben administrativen
und internen Buchungen enthält sie auch Codes, die nicht allein aufgrund ihres Formats ausgeschlossen werden können.

Als nicht reguläre Produktpositionen wurden folgende konkrete **StockCodes** identifiziert:

- `BANK CHARGES`: Bankgebühren
- `AMAZONFEE`: Amazon-Gebühren
- `POST`: Versandkosten
- `CRUK`: Provision für Cancer Research UK
- `DOT`: Versandkosten des Onlineshops
- `C2` und `C3`: Transportkosten
- `B`: Korrektur uneinbringlicher Forderungen
- `D`: Rabatte
- `m` und `M`: manuelle Buchungen
- `S`: Muster beziehungsweise kostenlose Proben

Darüber hinaus wurden mehrere Gruppen mit wechselnden **StockCodes**, aber gemeinsamen Präfixen festgestellt:

- `gift`: Gutscheine
- `test`: Testprodukte
- `adjust`: manuelle Anpassungen

Andere nicht numerisch beginnende **StockCodes** werden nicht pauschal ausgeschlossen, da das verwendete Format allein keine eindeutige Trennung
zwischen regulären Produkten und Sonderfällen ermöglicht.

## 7. Untersuchung negativer Mengen

Nach dem Entfernen der dokumentierten Stornierungen verbleiben Datensätze mit negativer **Quantity**. Da eine negative Menge grundsätzlich auch die Korrektur
einer früheren Verkaufsbuchung darstellen könnte, werden diese Datensätze näher untersucht.

In [ ]:
negative_quantity_mask = (
    ~distinct_rows["Invoice"].str.startswith("C")
    & distinct_rows["Quantity"].lt(0)
)

negative_quantity_rows = distinct_rows.loc[negative_quantity_mask]

print(f"Negative quantity rows: {negative_quantity_rows.shape[0]:,}")
print(negative_quantity_rows["Price"].value_counts(dropna=False))

### Beobachtungen

Nach dem Entfernen der Stornorechnungen verbleiben **3.393** Datensätze mit negativer **Quantity**.

Alle diese Datensätze besitzen einen **Price** von `0`. Es handelt sich somit nicht um reguläre Verkaufspositionen mit einem negativen Verkaufspreis.

In [ ]:
print(
    negative_quantity_rows["Description"]
    .fillna("<NA>")
    .str.strip()
    .value_counts()
)

Die häufigsten Beschreibungen enthalten Begriffe wie *damaged*, *missing*, *stock check*, *adjustment*, *wrong barcode* oder *thrown away*.
Diese deuten überwiegend auf interne Bestandskorrekturen, Inventurdifferenzen oder beschädigte Ware hin.

### Untersuchung möglicher Gegenbuchungen

Negative Mengen können grundsätzlich Rücknahmen oder Korrekturen früherer Verkaufsbuchungen darstellen. Deshalb wird untersucht, ob sich zu den negativen Positionen entsprechende positive Gegenbuchungen im Datensatz finden lassen.

Als Vergleichsmerkmale werden verwendet:

- **StockCode**
- absoluter Wert von **Quantity**
- **InvoiceDate**
- **Country**

In [ ]:
match_columns = ["StockCode", "Quantity", "InvoiceDate", "Country"]

negative_rows = negative_quantity_rows.copy()
negative_rows["Quantity"] = negative_rows["Quantity"].abs()

candidate_rows = distinct_rows.drop(negative_rows.index)

matched_rows = pd.merge(
    candidate_rows,
    negative_rows,
    on=match_columns,
    how="inner",
)

print(f"Negative quantity rows : {negative_rows.shape[0]:,}")
print(f"Potential matches      : {matched_rows.shape[0]:,}")

display(matched_rows)

Unter den gewählten Vergleichsmerkmalen konnten lediglich **6** potenzielle Gegenbuchungen identifiziert werden.

Die gefundenen Datensätze besitzen ebenfalls einen **Price** von `0` und keine **Customer ID**. Eine eindeutige Beziehung zwischen negativer und positiver
Buchung lässt sich daraus jedoch nicht ableiten. Dem Datensatz fehlt eine Referenz auf die ursprünglich korrigierte Rechnungsposition, sodass ein Matching
allein anhand gemeinsamer Merkmale keine belastbare Zuordnung ermöglicht.

### Schlussfolgerung

Die Untersuchung liefert keine ausreichenden Hinweise darauf, dass negative Mengen zuverlässig regulären Verkaufsbuchungen zugeordnet werden können.

Stattdessen deuten die Ergebnisse überwiegend auf interne Bestands- und Korrekturvorgänge hin.

## 8. Untersuchung von Nullpreis-Rechnungen

Im Datensatz treten Rechnungspositionen mit einem **Price** von `0` auf. Ein Nullpreis allein lässt jedoch nicht erkennen, ob es sich um eine kostenlose
Produktabgabe oder einen internen Vorgang handelt.

Für die Untersuchung werden vollständige Duplikate, dokumentierte Stornierungen und die zuvor untersuchten negativen Mengen ausgeschlossen. Der ursprüngliche
Datensatz wird dabei nicht verändert.

In [ ]:
null_price_base = distinct_rows.loc[
    ~distinct_rows["Invoice"].str.startswith("C")
    & ~negative_quantity_mask
]

### Rechnungen mit Nullpreispositionen

Zunächst werden alle Rechnungen identifiziert, die mindestens eine Position mit einem **Price** von `0` enthalten.

In [ ]:
price_zero = (
    null_price_base[null_price_base["Price"] == 0]
    .drop_duplicates(subset="Invoice")
)

invoice_rows = null_price_base[
    null_price_base["Invoice"].isin(price_zero["Invoice"])
]

print(f"Invoices containing Price == 0: {price_zero.shape[0]:,}")

Eine Rechnung mit einer Nullpreisposition kann gleichzeitig regulär bepreiste Positionen enthalten. Daher wird zusätzlich geprüft, welche der betroffenen Rechnungen
ausschließlich aus Positionen mit einem **Price** von `0` bestehen.

In [ ]:
free_invoice_summary = (
    invoice_rows.groupby("Invoice", as_index=False)["Price"]
    .agg(is_free_invoice=lambda group: group.eq(0).all())
)

free_invoices = free_invoice_summary.loc[
    free_invoice_summary["is_free_invoice"],
    "Invoice"
]

free_invoice_rows = invoice_rows[
    invoice_rows["Invoice"].isin(free_invoices)
]

print(f"Free invoices        : {free_invoices.nunique():,}")
print(f"Rows in free invoices: {free_invoice_rows.shape[0]:,}")

### Untersuchung der Beschreibungen

Um die ausschließlich aus Nullpreispositionen bestehenden Rechnungen näher einzuordnen, werden deren **Descriptions** untersucht.

Zunächst wird geprüft, wie viele dieser Positionen keine Beschreibung besitzen.

In [ ]:
missing_description_mask = free_invoice_rows["Description"].isna()

print(f"Rows without Description: {missing_description_mask.sum():,}")

Positionen ohne **Description** können anhand ihrer Produktbeschreibung nicht näher eingeordnet werden.

Die vorhandenen Beschreibungen werden von führenden und nachfolgenden Leerzeichen bereinigt und nach ihrer Häufigkeit gruppiert. Dadurch lassen sich wiederkehrende
Beschreibungen leichter untersuchen.

In [ ]:
description_counts = (
    free_invoice_rows.loc[~missing_description_mask, "Description"]
    .str.strip()
    .value_counts()
)

print(description_counts)

Die Ausgabe in Python reicht für eine vollständige fachliche Einordnung der Beschreibungen nicht aus. Daher wird die gruppierte Übersicht zusätzlich als CSV-Datei
exportiert und anschließend manuell in Excel untersucht.

In [ ]:
description_counts.to_csv(RAW_DATA_DIR / "free_invoice_descriptions.csv")

### Manuelle Untersuchung der Beschreibungen

Die manuelle Untersuchung der gruppierten Beschreibungen in Excel zeigt, dass unterschiedliche Arten von Nullpreispositionen vorkommen.

Neben regulären Produktbeschreibungen treten Beschreibungen auf, die auf interne Vorgänge wie Bestandskorrekturen, Prüfungen, Zuordnungen oder fehlerhafte Buchungen hinweisen.

Dabei wurden sowohl wiederkehrende Präfixe als auch einzelne interne Beschreibungen identifiziert.

In [ ]:
internal_description_prefixes = [
    "?sold",
    "add stock",
    "adjust",
    "allocate stock",
    "amazon",
    "check",
    "dotcom",
    "found",
    "incorrect",
    "mailout",
    "marked as",
    "sold ",
    "wrong",
]

exact_internal_descriptions = [
    "?",
    "22719",
    "alan hodge cant mamage this section",
    "amendment",
    "came coded as 20713",
    "correct previous adjustment",
    "damaged",
    "debenhams",
    "did  a credit  and did not tick ret",
    "eurobargain invc/credit",
    "fba",
    "for online retail orders",
    "had been put aside",
    "had been put aside.",
    "john lewis",
    "lighthouse trading zero invc incorr",
    "michel oops",
    "on cargo order",
    "rcvd be air temp fix for dotcom sit",
    "returned",
    "rust fixed",
    "sale error",
    "taig adjust",
    "temp",
    "test",
    "tk maxx mix up with pink",
    "update",
    "website fixed",
]

### Schlussfolgerung

Ein **Price** von `0` ist allein kein ausreichendes Kriterium, um eine Rechnungsposition als internen Vorgang einzuordnen. Die Untersuchung zeigt, dass neben eindeutig
internen Beschreibungen auch reguläre Produktbeschreibungen vorkommen.

Positionen ohne **Description** lassen sich anhand ihrer Beschreibung nicht weiter einordnen. Bei den vorhandenen Beschreibungen konnten durch die manuelle Untersuchung
sowohl eindeutig interne Bezeichnungen als auch reguläre Produktbeschreibungen identifiziert werden.